# Workshop 4 — Agents: Tool-Calling LLM Control

**Estimated time:** ~90 minutes  
**Prerequisites:** Workshop 1 (Workshops 2 & 3 are helpful but optional)

In this workshop you will:
1. Understand the ReAct (Reason + Act) agent loop
2. Build a tool-calling LLM agent that controls the robot
3. Implement two new tools: `get_scene_summary` and `check_held_object`
4. Debug common agent failure modes
5. Explore extensions: richer tools, vision-only agents, multi-agent pipelines

> ⚠️ **Before you begin:** Close any open simulator windows from previous workshops. Each notebook creates a fresh robot instance. Having two viewer tabs open simultaneously causes confusion.

## Section 1 — Quick Setup

In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv requests

In [ ]:
import asyncio
import os
import re
import json
import math
import base64
import time
import requests

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── Key loader: tries .env first, then Colab Secrets ────────────────────────
def _load_keys():
    # Mode B: local .env file
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # Mode A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for the agent LLM)'}")

In [ ]:
# Same setup as Workshop 1
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop4-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> Open the viewer URL in **Google Chrome**, wait for the scene to load, then run the next cell.

In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """Poll until the viewer joins and skills become available."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")

## Section 2 — Why Tool-Calling Agents?

Hand-coded robot logic fails at scale: the number of if/else branches grows exponentially with the number of states and tasks. LLM agents offer a different approach.

### The ReAct Loop

```
User Task
   ↓
[LLM THINK] → JSON tool call
   ↓
[EXECUTE] → SDK invoke → robot action
   ↓
[OBSERVE] → result string appended to history
   ↓
[LLM THINK] → next call or "done"
```

**Key properties:**
- The LLM only sees text — we encode robot state as text (tool results)
- Every request includes the full conversation history (stateless API)
- The agent is bounded by a turn budget to prevent infinite loops
- Tools are the interface between language and physics

## Section 3 — LLM and VLM Helpers

In [ ]:
TOKAMAK_URL = "https://api.tokamak.sh/v1/chat/completions"

def call_llm(messages, model="minimaxai/minimax-m2.7"):
    """Send messages to Tokamak LLM API and return text reply."""
    response = requests.post(
        TOKAMAK_URL,
        headers={
            "Authorization": f"Bearer {TOKAMAK_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"model": model, "messages": messages},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def ask_vlm(jpeg_bytes, prompt, model="qwen/qwen3.6-35b-a3b"):
    """Send an image + text prompt to a vision-language model."""
    b64_image = base64.b64encode(jpeg_bytes).decode("utf-8")
    image_url = f"data:image/jpeg;base64,{b64_image}"
    messages = [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": prompt},
        ],
    }]
    return call_llm(messages, model=model)


# Quick test
reply = call_llm([{"role": "user", "content": "Say 'LLM ready' and nothing else."}])
print(f"LLM test: {reply}")

## Section 4 — The Tool Registry

A tool registry maps tool names to their descriptions. The LLM reads these descriptions to decide which tool to call.

**What makes a good tool description:**
- Precise argument names with units (e.g., `vx in m/s, max 1.5`)
- Clear preconditions (e.g., `must be within 1m of target`)
- Expected return value (e.g., `returns cube id and color`)
- Any known quirks (e.g., `cannot spin in place`)

In [ ]:
TOOLS = {
    "set_velocity": {
        "description": (
            "Move the robot for a fixed duration. "
            "Args: vx (forward m/s, max 1.5), vy (sideways m/s), "
            "wz (turn rad/s, max 0.6), duration_s. "
            "IMPORTANT: cannot spin in place — always pair wz with vx=0.2."
        )
    },
    "go_to": {
        "description": (
            "Walk to a named entity using built-in pathfinding. "
            "Args: entity_id (e.g. 'pad_C', 'cube_2'). "
            "Pads: pad_A, pad_B, pad_C, pad_D, pad_E."
        )
    },
    "pick_cube": {
        "description": (
            "Pick up the nearest visible cube on the floor. "
            "No args. Returns the cube id and its color. "
            "Precondition: must be within ~1m of a cube (use go_to first)."
        )
    },
    "look": {
        "description": (
            "Capture the robot's POV camera and describe what it sees. "
            "No args. Returns a text description of the scene."
        )
    },
    "place": {
        "description": (
            "Deliver the currently held cube to its correct pad. "
            "Routing rules: red→pad_B, green→pad_C, blue→pad_D, yellow→pad_E. "
            "No args — routing is automatic based on held cube color. "
            "WARNING: delivering to the wrong pad ends the benchmark round."
        )
    },
    "get_scene_summary": {
        "description": (
            "Get a structured summary of all visible cubes from scene state: "
            "their IDs, colors, and distances from the robot. "
            "No args. Use this to plan which cube to pick next."
        )
    },
    "check_held_object": {
        "description": (
            "Check what the robot is currently holding. "
            "No args. Returns the held entity id and color, or 'nothing' if hands are empty."
        )
    },
}

print(f"Tool registry: {list(TOOLS.keys())}")

## Section 5 — System Prompt Builder

The system prompt is the LLM's instruction manual. It describes the task format, the tool calling convention, and lists all available tools.

In [ ]:
def build_system_prompt(tools):
    """Build a system prompt that advertises all tools in JSON call format."""
    lines = [
        "You control a humanoid warehouse robot by calling tools.",
        "To call a tool, reply with ONLY a single JSON object in a fenced code block:",
        "```json",
        '{"tool": "<tool name>", "args": {...}}',
        "```",
        "Write nothing else outside the code block.",
        "When the task is complete (or impossible), call the special tool 'done':",
        '{"tool": "done", "args": {"summary": "<one sentence>"}}',
        "",
        "Available tools:",
    ]
    for name, spec in tools.items():
        lines.append(f"- {name}: {spec['description']}")
    return "\n".join(lines)


# Print the prompt so you can read what the model sees
print(build_system_prompt(TOOLS))

> **Tip:** Small wording changes in the system prompt can have large behavioral effects. Try tweaking a tool description and observe how the agent's strategy changes.

## Section 6 — Tool Call Parser

In [ ]:
def parse_tool_call(reply):
    """Extract and parse the JSON tool call from the model's reply."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", reply, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    raise ValueError(f"Could not parse tool call from: {reply[:200]}")


def safe_parse_tool_call(reply):
    """Like parse_tool_call but returns an error dict instead of raising."""
    try:
        return parse_tool_call(reply)
    except Exception as e:
        return {"tool": "error", "args": {"message": str(e), "raw_reply": reply[:200]}}


# Demo on a valid reply
valid_reply = '```json\n{"tool": "go_to", "args": {"entity_id": "pad_C"}}\n```'
print("Valid:", parse_tool_call(valid_reply))

# Demo on an invalid reply
bad_reply = "I think we should go to pad_C. Let me navigate there."
print("Invalid:", safe_parse_tool_call(bad_reply))

## Section 7 — Tool Executor

The executor dispatches each tool call to the actual SDK. It bridges the language layer and the physical robot.

In [ ]:
async def execute_tool(name, args):
    """Execute a tool call and return a text result string."""

    if name == "set_velocity":
        result = await session.invoke("set_velocity", args, timeout_s=60)
        return f"set_velocity finished: status={result.status}"

    if name == "go_to":
        result = await session.invoke(
            "go_to",
            {"target": {"kind": "entity", "entity_id": args["entity_id"]}},
            timeout_s=300,
        )
        if result.status != "done":
            err = f" ({result.error.message})" if result.error else ""
            return f"go_to failed{err}"
        return f"Reached {args['entity_id']}"

    if name == "look":
        jpeg = await session.get_vision("pov")
        return ask_vlm(jpeg, "Describe what you see: objects, colors, positions, anything relevant for warehouse sorting.")

    if name == "pick_cube":
        scene  = await session.state.get("scene_state")
        status = await session.state.get("robot_status")
        rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
        visible = [
            (eid, e) for eid, e in scene.entities.items()
            if eid.startswith("cube_") and e.visible and e.attached_to is None
        ]
        if not visible:
            return "No visible cubes to pick."
        nearest_id, nearest_e = min(
            visible,
            key=lambda x: math.hypot(x[1].pose.position[0]-rx, x[1].pose.position[1]-ry)
        )
        result = await session.invoke(
            "pick_entity",
            {"target": {"kind": "entity", "entity_id": nearest_id}},
            timeout_s=300,
        )
        if result.status != "done":
            return f"pick_entity failed: {result.error.message if result.error else 'unknown'}"
        color = nearest_e.state.get("color", "unknown")
        return f"Picked {nearest_id} ({color})"

    if name == "place":
        status = await session.state.get("robot_status")
        scene  = await session.state.get("scene_state")
        held = [
            eid for eid, e in scene.entities.items()
            if e.attached_to == status.robot.entity_id
        ]
        if not held:
            return "Not holding anything."
        cube_color = scene.entities[held[0]].state.get("color", "")
        PAD_MAP = {"red": "pad_B", "green": "pad_C", "blue": "pad_D", "yellow": "pad_E"}
        pad_id = PAD_MAP.get(cube_color)
        if not pad_id:
            return f"Unknown color '{cube_color}' — cannot determine pad."
        result = await session.invoke(
            "place_entity",
            {"target": {"kind": "entity", "entity_id": pad_id}},
            timeout_s=300,
        )
        if result.status != "done":
            return "place_entity failed"
        return f"Placed {cube_color} cube on {pad_id}"

    if name == "get_scene_summary":
        scene  = await session.state.get("scene_state")
        status = await session.state.get("robot_status")
        rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
        lines = []
        for eid, e in scene.entities.items():
            if eid.startswith("cube_") and e.visible and e.attached_to is None:
                dist  = math.hypot(e.pose.position[0]-rx, e.pose.position[1]-ry)
                color = e.state.get("color", "unknown")
                lines.append(f"  {eid}: {color}, {dist:.1f}m away")
        return ("Visible cubes:\n" + "\n".join(lines)) if lines else "No visible cubes."

    if name == "check_held_object":
        status = await session.state.get("robot_status")
        scene  = await session.state.get("scene_state")
        held = [
            eid for eid, e in scene.entities.items()
            if e.attached_to == status.robot.entity_id
        ]
        if not held:
            return "Holding: nothing"
        color = scene.entities[held[0]].state.get("color", "unknown")
        return f"Holding: {held[0]} ({color})"

    if name == "done":
        return f"[done] {args.get('summary', '')}"

    if name == "error":
        return f"[parse error] {args.get('message', '')}"

    return f"ERROR: unknown tool '{name}'"


print("execute_tool defined.")

## Section 8 — The Agent Loop

In [ ]:
async def run_agent(task, max_turns=8):
    """Run the ReAct agent loop for a given task."""
    messages = [
        {"role": "system",  "content": build_system_prompt(TOOLS)},
        {"role": "user",    "content": task},
    ]
    tool_log = []

    for turn in range(1, max_turns + 1):
        reply = call_llm(messages)                      # THINK
        messages.append({"role": "assistant", "content": reply})

        call      = safe_parse_tool_call(reply)
        tool_name = call["tool"]
        tool_args = call.get("args", {})

        history_chars = sum(len(str(m["content"])) for m in messages)
        print(f"turn {turn} | tool={tool_name} | args={tool_args} | history~{history_chars:,}chars")

        if tool_name == "done":
            print(f"\n✓ Agent done: {tool_args.get('summary', '')}")
            break

        result_text = await execute_tool(tool_name, tool_args)  # ACT
        tool_log.append({"turn": turn, "tool": tool_name, "result": result_text[:100]})
        print(f"  → {result_text[:120]}")

        messages.append({"role": "user", "content": result_text})  # OBSERVE

    return messages, tool_log


# Demo run
print("Running agent: pick and deliver the nearest cube...")
messages, tool_log = await run_agent(
    "Use get_scene_summary to find a cube, go to it, pick it up, and deliver it to the correct pad.",
    max_turns=10,
)

## Section 9 — Debugging Agents

When an agent misbehaves, inspect the full message history and tool log.

In [ ]:
# Print full message history from the last run
print(f"Total messages: {len(messages)}")
for i, m in enumerate(messages):
    role    = m["role"]
    content = str(m["content"])
    preview = content[:120].replace("\n", " ")
    print(f"  [{i}] {role}: {preview}")

In [ ]:
# Print tool call log as a table
print(f"{'Turn':<6} {'Tool':<22} {'Result (truncated)'}")
print("-" * 70)
for entry in tool_log:
    print(f"  {entry['turn']:<4} {entry['tool']:<22} {entry['result'][:40]}")

### Common Agent Failure Modes

| Symptom | Cause | Fix |
|---|---|---|
| Agent calls `go_to` with a non-existent entity ID | Model hallucinates entity names | Add `get_scene_summary` and instruct agent to call it first |
| Agent loops calling the same tool repeatedly | No convergence signal | Add turn budget; tool results should say "already done" |
| Agent delivers to wrong pad | Model doesn't know color→pad rules | Encode routing in `place` tool description |
| `parse_tool_call` fails | Model generates free text instead of JSON | Add `safe_parse_tool_call`; reinforce JSON-only constraint in system prompt |

---
## Exercises

### Exercise 1 — Add a `spin` Tool

Add a `spin` tool to `TOOLS` that makes the robot do a full 360° rotation:
- `set_velocity(wz=0.5, vx=0.2, duration_s=12.5)` covers ~360° at 0.5 rad/s

Implement it in `execute_tool`, then run the agent with the task: "Survey the scene by spinning, then tell me what colors of cubes you see."

In [ ]:
## Exercise 1: Add spin tool

# 1. Add to TOOLS:
# TOOLS["spin"] = {"description": "..."}

# 2. Add to execute_tool (or define a new version):
# if name == "spin":
#     ...

# 3. Run the agent:
# YOUR CODE HERE

# Expected: agent calls spin, then look, then done
# Output includes a description of visible cube colors after 360° survey

### Exercise 2 — Add a Pad Map to the System Prompt

Modify `build_system_prompt` to append a "Map of pads:" section listing each pad's name and approximate x,y position (get these from `scene_state`). Does the agent make fewer navigation mistakes?

In [ ]:
## Exercise 2: Add pad map to system prompt

# 1. Read pad positions:
# scene = await session.state.get("scene_state")
# pad_map_text = "Map of pads:\n"
# for eid, e in scene.entities.items():
#     if eid.startswith("pad_"): ...

# 2. Modify build_system_prompt to append pad_map_text

# 3. Run the agent on: "Sort all visible cubes to their correct pads."

# YOUR CODE HERE

# Expected: agent uses pad positions to navigate more efficiently

### Exercise 3 — Vision-Only Tools

Add a `perceive` tool that uses OpenCV-based HSV segmentation (no `scene_state`). Copy the `perceive()` function inline.

Run the agent with only vision tools: `{"set_velocity", "perceive", "pick_cube", "place", "done"}`. Remove `go_to` and `get_scene_summary`. What breaks?

In [ ]:
## Exercise 3: Vision-only agent
import cv2
import numpy as np

# Copy perceive() from Workshop 2 here:
async def perceive(session):
    # YOUR CODE HERE (paste the full perceive() function from Workshop 2)
    pass

# Add perceive to TOOLS, remove go_to and get_scene_summary
# TOOLS_VISION = {k: v for k, v in TOOLS.items() if k not in ("go_to", "get_scene_summary")}
# TOOLS_VISION["perceive"] = {"description": "..."}

# YOUR CODE HERE

# Expected: agent attempts the task but may struggle without go_to
# Observe: what does it do differently? Does it succeed? What fails first?

### Exercise 4 — Instrument the Agent Loop

Modify `run_agent` (or write a wrapper) to count the number of calls per tool type. After a run, print a summary table like:
```
go_to:              3 calls
pick_cube:          2 calls
get_scene_summary:  1 call
place:              2 calls
done:               1 call
```

In [ ]:
## Exercise 4: Count tool calls per type
from collections import Counter

# YOUR CODE HERE

# Expected output (example):
# get_scene_summary:  1 calls
# go_to:              3 calls
# pick_cube:          2 calls
# place:              2 calls
# done:               1 calls

### Exercise 5 — Degraded Tool Set

Remove `go_to` and `get_scene_summary` from the tool registry, leaving: `set_velocity`, `pick_cube`, `look`, `place`, `check_held_object`, `done`. Run the sorting task 3 times and record success/failure.

In [ ]:
## Exercise 5: Degraded tool set — 3 runs
TOOLS_DEGRADED = {k: v for k, v in TOOLS.items() if k not in ("go_to", "get_scene_summary")}

# YOUR CODE HERE — run 3 times, record results

# Results table:
# Run 1: [success / fail / partial]
# Run 2: ...
# Run 3: ...
# Observation: ...

### Exercise 6 (Stretch) — Two-Agent Pipeline

Build a Planner + Executor pipeline:
1. **Planner:** call `call_llm` with the scene summary and ask it to return a JSON list of steps: `[{"action": "go_to", "target": "cube_2"}, {"action": "pick"}, {"action": "go_to", "target": "pad_C"}, {"action": "place"}, ...]`
2. **Executor:** iterate over the plan and call the SDK directly for each step (no LLM involved)

This separates high-level planning (LLM strength) from reliable execution (SDK strength).

In [ ]:
## Exercise 6 (stretch): Two-agent Planner + Executor

# --- Planner ---
async def make_plan(session):
    """Ask LLM to produce a step-by-step JSON plan given the scene summary."""
    # 1. Get scene summary text
    # 2. Build a prompt asking for JSON list of steps
    # 3. Parse the JSON
    # YOUR CODE HERE
    pass


# --- Executor ---
async def execute_plan(session, plan):
    """Execute each step in the plan as direct SDK calls."""
    for step in plan:
        action = step.get("action")
        target = step.get("target", "")
        print(f"Executing: {action} {target}")
        # YOUR CODE HERE — handle go_to, pick, place actions


# Run it:
# plan = await make_plan(session)
# print("Plan:", plan)
# await execute_plan(session, plan)

---
## Cleanup

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")